In [11]:
import pandas as pd

df = pd.read_csv("data/nt_split_column_df.csv")
df.head()

,verse_ref,text,book,chapter,verse
0,1 corinthians 1:1,Παῦλος κλητὸς ἀπόστολος Χριστοῦ Ἰησοῦ διὰ θελή...,1 corinthians,1,1
1,1 corinthians 1:2,"τῇ ἐκκλησίᾳ τοῦ θεοῦ τῇ οὔσῃ ἐν Κορίνθῳ, ἡγι...",1 corinthians,1,2
2,1 corinthians 1:3,χάρις ὑμῖν καὶ εἰρήνη ἀπὸ θεοῦ πατρὸς ἡμῶν καὶ...,1 corinthians,1,3
3,1 corinthians 1:4,Εὐχαριστῶ τῷ θεῷ μου πάντοτε περὶ ὑμῶν ἐπὶ τῇ...,1 corinthians,1,4
4,1 corinthians 1:5,ὅτι ἐν παντὶ ἐπλουτίσθητε ἐν αὐτῷ ἐν παντὶ λόγ...,1 corinthians,1,5


In [12]:
import re
import unicodedata
from collections import Counter


def strip_diacritics(text):
    decomposed = unicodedata.normalize("NFD", text)
    return "".join(char for char in decomposed if unicodedata.category(char) != "Mn")


RAW_GREEK_STOPWORDS = {
    "και", "δε", "γαρ", "ουν", "ως", "μη", "ου", "ουκ", "ουχ",
    "ο", "η", "το", "οι", "αι", "τα",
    "του", "της", "των", "τω", "τη", "τον", "την", "τους", "τας",
    "τοις", "ταις",
    "εν", "εις", "εκ", "εξ", "επι", "δια", "κατα", "προς", "υπο",
    "υπερ", "μετα", "παρα", "απο", "περι",
    "οτι", "ινα", "εαν", "αν",
    "τις", "τι", "τινα", "τινες",
    "αυτου", "αυτων", "αυτον", "αυτη", "αυτης", "αυται", "αυτοι", "αυτω",
    "ημιν", "ημων", "ημας", "υμιν", "υμων", "υμας",
    "μου", "σου", "με", "σε"
}
GREEK_STOPWORDS = {
    strip_diacritics(word.lower()).replace("ς", "σ") for word in RAW_GREEK_STOPWORDS
}

RAW_STEM_SUFFIXES = {
    "ομεθα", "ουμεθα", "εσθαι", "ησονται", "ονται", "ουσιν", "ουσι",
    "ομεν", "ουμεν", "ειτε", "ετε", "εται", "ειται", "ειν", "εις",
    "οντες", "οντας", "μενος", "μενη", "μενον", "ματος", "ματων",
    "μασι", "σεως", "σεων", "σεσι", "τητες", "τητων", "τητα",
    "ικος", "ικου", "ικον", "ικοι", "ικων",
    "οις", "αις", "ους", "ων", "ου", "ος", "ον", "οι", "αι",
    "ας", "ες", "ης", "ην", "ει"
}
STEM_SUFFIXES = sorted(
    {suffix.replace("ς", "σ") for suffix in RAW_STEM_SUFFIXES},
    key=len,
    reverse=True,
)


def normalize_greek(text):
    text = strip_diacritics(str(text).lower()).replace("ς", "σ")
    text = re.sub(r"[^α-ω\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def light_stem_greek_token(token):
    if len(token) <= 4:
        return token

    for suffix in STEM_SUFFIXES:
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            return token[:-len(suffix)]

    return token


def preprocess_document(text):
    normalized = normalize_greek(text)
    tokens = [
        token
        for token in normalized.split()
        if len(token) > 2 and token not in GREEK_STOPWORDS
    ]
    stems = [light_stem_greek_token(token) for token in tokens]
    stems = [stem for stem in stems if len(stem) > 1]
    return normalized, tokens, stems


df[["normalized_text", "tokens", "stemmed_tokens"]] = df["text"].apply(
    lambda text: pd.Series(preprocess_document(text))
)
corpus = df["stemmed_tokens"].tolist()
df["corpus_text"] = df["stemmed_tokens"].str.join(" ")

top_stems = Counter(stem for doc in corpus for stem in doc).most_common(20)
pd.DataFrame(top_stems, columns=["stem", "count"])

,stem,count
0,αυτ,1304
1,εστιν,934
2,λεγ,867
3,ιησ,859
4,θεου,704
5,ειπεν,626
6,κυρι,527
7,ανθρωπ,525
8,αλλ,470
9,χριστ,469


In [13]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=5,
    max_df=0.45,
    max_features=3000,
)

doc_term_matrix = vectorizer.fit_transform(df["corpus_text"])
feature_names = vectorizer.get_feature_names_out()

print(doc_term_matrix.shape)
df[["verse_ref", "stemmed_tokens", "corpus_text"]].head()

(7958, 2422)


,verse_ref,stemmed_tokens,corpus_text
0,1 corinthians 1:1,"[παυλ, κλητ, αποστολ, χριστ, ιησ, θελη, θεου, ...",παυλ κλητ αποστολ χριστ ιησ θελη θεου σωσθεν α...
1,1 corinthians 1:2,"[εκκλησια, θεου, ουση, κορινθω, ηγιασμεν, χρισ...",εκκλησια θεου ουση κορινθω ηγιασμεν χριστω ιησ...
2,1 corinthians 1:3,"[χαρισ, ειρηνη, θεου, πατρ, κυρι, ιησ, χριστ]",χαρισ ειρηνη θεου πατρ κυρι ιησ χριστ
3,1 corinthians 1:4,"[ευχαριστω, θεω, παντοτε, χαριτι, θεου, δοθεισ...",ευχαριστω θεω παντοτε χαριτι θεου δοθειση χρισ...
4,1 corinthians 1:5,"[παντι, επλουτισθητε, παντι, λογω, παση, γνωσ]",παντι επλουτισθητε παντι λογω παση γνωσ


In [14]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 10
lda = LatentDirichletAllocation(
    n_components=n_topics,
    learning_method="batch",
    max_iter=15,
    random_state=42,
)

topic_matrix = lda.fit_transform(doc_term_matrix)
df["dominant_topic"] = topic_matrix.argmax(axis=1)


def describe_topics(model, feature_names, n_words=10):
    rows = []
    for topic_index, weights in enumerate(model.components_):
        top_indices = weights.argsort()[-n_words:][::-1]
        rows.append(
            {
                "topic": topic_index,
                "top_terms": ", ".join(feature_names[i] for i in top_indices),
            }
        )
    return pd.DataFrame(rows)


describe_topics(lda, feature_names, n_words=12)

,topic,top_terms
0,0,"αυτ, εγενετο, εστ, ημερ, ουραν, γησ, ειτε, εκε..."
1,1,"αυτ, ουτ, εστιν, λεγ, κυρι, ιουδαι, ιησ, οφθαλ..."
2,2,"εστιν, θεου, χριστ, κυρι, ιησ, λογ, πνευμα, αν..."
3,3,"αυτ, αδελφ, λεγ, πρωτ, πολλ, ημερ, κυρι, ιδου,..."
4,4,"ιησ, λεγ, ειπεν, αυτ, μαθητ, αμην, πετρ, λεγω,..."
5,5,"χριστ, αλλα, θεου, υμεισ, αλλ, παντ, θεοσ, εαυ..."
6,6,"ανθρωπ, αγι, αυτ, ουτε, πνευ, παντ, λεγ, πνευμ..."
7,7,"αυτ, ειπεν, εαυτ, πασ, σοι, μοι, λεγ, αδελφ, μ..."
8,8,"αυτ, παντα, εγω, ταυτα, εστιν, μοι, αλλ, ειπεν..."
9,9,"αγγελ, μεγαλη, ημερα, μηδε, γην, εχοντ, εκεινη..."


In [15]:
print("Sample normalized text:\n", df.loc[0, "normalized_text"][:180])
print("\nSample stems:\n", df.loc[0, "stemmed_tokens"][:20])
print("\nTop 10 corpus stems:")
for stem, count in Counter(stem for doc in corpus for stem in doc).most_common(10):
    print(f"{stem}: {count}")

print("\nTopic preview:")
for row in describe_topics(lda, feature_names, n_words=8).head(5).itertuples(index=False):
    print(f"Topic {row.topic}: {row.top_terms}")

Sample normalized text:
 παυλοσ κλητοσ αποστολοσ χριστου ιησου δια θεληματοσ θεου και σωσθενησ ο αδελφοσ

Sample stems:
 ['παυλ', 'κλητ', 'αποστολ', 'χριστ', 'ιησ', 'θελη', 'θεου', 'σωσθεν', 'αδελφ']

Top 10 corpus stems:
αυτ: 1304
εστιν: 934
λεγ: 867
ιησ: 859
θεου: 704
ειπεν: 626
κυρι: 527
ανθρωπ: 525
αλλ: 470
χριστ: 469

Topic preview:
Topic 0: αυτ, εγενετο, εστ, ημερ, ουραν, γησ, ειτε, εκει
Topic 1: αυτ, ουτ, εστιν, λεγ, κυρι, ιουδαι, ιησ, οφθαλμ
Topic 2: εστιν, θεου, χριστ, κυρι, ιησ, λογ, πνευμα, ανθρωπ
Topic 3: αυτ, αδελφ, λεγ, πρωτ, πολλ, ημερ, κυρι, ιδου
Topic 4: ιησ, λεγ, ειπεν, αυτ, μαθητ, αμην, πετρ, λεγω


In [ ]:
candidate_topics = [5, 8, 10, 12, 15, 20, 25, 30, 40, 50]
lda_results = []
lda_models = {}

for k in candidate_topics:
    model = LatentDirichletAllocation(
        n_components=k,
        learning_method="batch",
        max_iter=8,
        random_state=42,
    )
    doc_topic = model.fit_transform(doc_term_matrix)
    lda_models[k] = (model, doc_topic)
    lda_results.append(
        {
            "n_topics": k,
            "perplexity": model.perplexity(doc_term_matrix),
            "log_likelihood": model.score(doc_term_matrix),
            "avg_topic_confidence": doc_topic.max(axis=1).mean(),
        }
    )

lda_eval_df = pd.DataFrame(lda_results).sort_values("perplexity")
print(lda_eval_df)

In [20]:
print(lda_eval_df)

   n_topics   perplexity  log_likelihood  avg_topic_confidence
0         5  1306.368387  -442934.666583              0.711946
1         8  1380.309203  -446333.467656              0.682188
2        10  1443.359293  -449090.815611              0.670214
3        12  1479.932533  -450635.576644              0.649557
4        15  1513.637835  -452025.767656              0.622677
5        20  1626.679969  -456472.098916              0.605668
6        25  1727.750713  -460193.320177              0.590280
7        30  1777.040482  -461929.803626              0.572857
8        40  1990.487560  -468932.195336              0.552086
9        50  2135.232858  -473265.610812              0.532171


In [17]:
best_k = int(lda_eval_df.iloc[0]["n_topics"])
best_lda, best_topic_matrix = lda_models[best_k]

print(f"Best topic count by perplexity: {best_k}")

best_topics_df = describe_topics(best_lda, feature_names, n_words=12)
df["best_dominant_topic"] = best_topic_matrix.argmax(axis=1)
df["best_topic_confidence"] = best_topic_matrix.max(axis=1)

best_topics_df

Best topic count by perplexity: 5


,topic,top_terms
0,0,"εαυτ, αλλα, παντ, μεν, αυτ, πασ, υμεισ, ουτωσ,..."
1,1,"αυτ, εστιν, ανθρωπ, κυρι, ουτ, λεγ, παντ, ειπε..."
2,2,"εστιν, θεου, χριστ, αλλ, ιησ, κυρι, παντα, ανθ..."
3,3,"αυτ, λεγ, ιδου, πολλ, αγγελ, ανθρωπ, ημερ, παν..."
4,4,"ιησ, ειπεν, αυτ, λεγ, εγω, μαθητ, πετρ, ειμι, ..."


In [8]:
topic_to_inspect = int(best_topics_df.iloc[0]["topic"])
(
    df[df["best_dominant_topic"] == topic_to_inspect]
    .sort_values("best_topic_confidence", ascending=False)
    [["verse_ref", "text", "best_topic_confidence"]]
    .head(10)
)

,verse_ref,text,best_topic_confidence
18832,2 Kings 25:27,Καὶ ἐγενήθη ἐν τῷ τριακοστῷ καὶ ἑβδόμῳ ἔτει τῆ...,0.965006
21277,Ezra-Nehemiah 10:9,Καὶ συνήχθησαν πάντες ἄνδρες Ιουδα καὶ Βενιαμι...,0.962997
35916,Jeremiah 52:31,Καὶ ἐγένετο ἐν τῷ τριακοστῷ καὶ ἑβδόμῳ ἔτει ἀπ...,0.961686
17547,1 Kings 7:13,"καὶ δώδεκα βόες ὑποκάτω τῆς θαλάσσης, οἱ τρεῖς...",0.961680
21164,Ezra-Nehemiah 5:14,καὶ τὰ σκεύη τοῦ οἴκου τοῦ θεοῦ τὰ χρυσᾶ καὶ τ...,0.961453
37517,Ezekiel 48:1,Καὶ ταῦτα τὰ ὀνόματα τῶν φυλῶν· ἀπὸ τῆς ἀρχῆς ...,0.961084
17102,2 Samuel 19:9,"καὶ ἀνέστη ὁ βασιλεὺς καὶ ἐκάθισεν ἐν τῇ πύλῃ,...",0.960934
18997,1 Chronicles 4:41,καὶ ἤλθοσαν οὗτοι οἱ γεγραμμένοι ἐπ’ ὀνόματος ...,0.960668
38066,Daniel (Theodotion) 1:2,καὶ ἔδωκεν κύριος ἐν χειρὶ αὐτοῦ τὸν Ιωακιμ βα...,0.960562
23439,1 Maccabees 11:34,ἑστάκαμεν αὐτοῖς τά τε ὅρια τῆς Ιουδαίας καὶ τ...,0.960382


In [9]:
# 1) Top words per topic
best_topics_df

# 2) Top verses for every topic
for t in sorted(df["best_dominant_topic"].unique()):
    print("\n" + "=" * 80)
    print(f"TOPIC {t}")
    print(best_topics_df.loc[best_topics_df["topic"] == t, "top_terms"].iloc[0])

    sample = (
        df[df["best_dominant_topic"] == t]
        .sort_values("best_topic_confidence", ascending=False)
        [["verse_ref", "text", "best_topic_confidence"]]
        .head(8)
    )
    print(sample.to_string(index=False))

# 3) Topic prevalence
topic_sizes = df["best_dominant_topic"].value_counts().sort_index()
print("\nTopic sizes:")
print(topic_sizes)

# 4) Confidence diagnostics
print("\nConfidence summary:")
print(df["best_topic_confidence"].describe())


TOPIC 0
βασιλευσ, βασιλεωσ, εωσ, οικ, αυτ, ιερουσαλημ, ημερ, βασιλ, πασ, βασιλεα, πολ, παντ
         verse_ref                                                                                                                                                                                                                                                                                           text  best_topic_confidence
     2 Kings 25:27       Καὶ ἐγενήθη ἐν τῷ τριακοστῷ καὶ ἑβδόμῳ ἔτει τῆς ἀποικεσίας τοῦ Ιωακιμ βασιλέως Ιουδα ἐν τῷ δωδεκάτῳ μηνὶ ἑβδόμῃ καὶ εἰκάδι τοῦ μηνὸς ὕψωσεν Ευιλμαρωδαχ βασιλεὺς Βαβυλῶνος ἐν τῷ ἐνιαυτῷ τῆς βασιλείας αὐτοῦ τὴν κεφαλὴν Ιωακιμ βασιλέως Ιουδα καὶ ἐξήγαγεν αὐτὸν ἐξ οἴκου φυλακῆς αὐτοῦ               0.965006
Ezra-Nehemiah 10:9                                                          Καὶ συνήχθησαν πάντες ἄνδρες Ιουδα καὶ Βενιαμιν εἰς Ιερουσαλημ εἰς τὰς τρεῖς ἡμέρας, οὗτος ὁ μὴν ὁ ἔνατος· ἐν εἰκάδι τοῦ μηνὸς ἐκάθισεν πᾶς ὁ λαὸς ἐν πλατείᾳ οἴκου τοῦ θεοῦ 

In [10]:
best_topics_df.head()

,topic,top_terms
0,0,"βασιλευσ, βασιλεωσ, εωσ, οικ, αυτ, ιερουσαλημ,..."
1,1,"θεοσ, αυτ, μεσ, κυρι, ημερ, χειρ, δικαι, γησ, ..."
2,2,"κυρι, αυτ, λεγ, ισραηλ, εστ, κυριω, προσωπ, ημ..."
3,3,"αυτ, ειπεν, κυρι, ισραηλ, υιοι, δαυιδ, υιοσ, α..."
4,4,"εστιν, αυτ, σοι, ανθρωπ, εγω, θεου, κυρι, λεγ,..."


In [21]:
# Hierarchical topic model: split each parent topic into subtopics.
parent_topic_col = "best_dominant_topic"
parent_conf_col = "best_topic_confidence"

target_subtopics_per_parent = 4
min_docs_to_split = 120
min_docs_per_subtopic = 60
max_subtopics_per_parent = 6
subtopic_random_state = 42

parent_topic_terms = best_topics_df.set_index("topic")["top_terms"].to_dict()

df["subtopic"] = 0
df["subtopic_confidence"] = df[parent_conf_col]
df["topic_path"] = df[parent_topic_col].astype(int).astype(str) + ".0"

subtopic_models = {}
subtopic_rows = []

for parent_topic in sorted(df[parent_topic_col].unique()):
    mask = df[parent_topic_col] == parent_topic
    parent_idx = df.index[mask].to_numpy()
    n_docs = parent_idx.size

    if n_docs < min_docs_to_split:
        subtopic_rows.append(
            {
                "parent_topic": int(parent_topic),
                "subtopic": 0,
                "topic_path": f"{int(parent_topic)}.0",
                "doc_count": int(n_docs),
                "avg_confidence": float(df.loc[mask, parent_conf_col].mean()),
                "top_terms": parent_topic_terms.get(int(parent_topic), ""),
                "split_applied": False,
            }
        )
        continue

    k_by_size = max(2, n_docs // min_docs_per_subtopic)
    n_subtopics = min(max_subtopics_per_parent, target_subtopics_per_parent, k_by_size)
    n_subtopics = min(n_subtopics, max(2, n_docs - 1))

    parent_doc_term_matrix = doc_term_matrix[parent_idx]
    sub_lda = LatentDirichletAllocation(
        n_components=n_subtopics,
        learning_method="batch",
        max_iter=15,
        random_state=subtopic_random_state,
    )
    sub_doc_topic = sub_lda.fit_transform(parent_doc_term_matrix)

    sub_ids = sub_doc_topic.argmax(axis=1)
    sub_conf = sub_doc_topic.max(axis=1)

    df.loc[parent_idx, "subtopic"] = sub_ids
    df.loc[parent_idx, "subtopic_confidence"] = sub_conf
    df.loc[parent_idx, "topic_path"] = [f"{int(parent_topic)}.{int(s)}" for s in sub_ids]

    subtopic_models[int(parent_topic)] = {
        "model": sub_lda,
        "doc_indices": parent_idx,
        "doc_topic": sub_doc_topic,
    }

    subtopic_terms_df = describe_topics(sub_lda, feature_names, n_words=10)
    for row in subtopic_terms_df.itertuples(index=False):
        sub_mask = sub_ids == row.topic
        subtopic_rows.append(
            {
                "parent_topic": int(parent_topic),
                "subtopic": int(row.topic),
                "topic_path": f"{int(parent_topic)}.{int(row.topic)}",
                "doc_count": int(sub_mask.sum()),
                "avg_confidence": float(sub_conf[sub_mask].mean()),
                "top_terms": row.top_terms,
                "split_applied": True,
            }
        )

subtopic_summary_df = (
    pd.DataFrame(subtopic_rows)
    .sort_values(["parent_topic", "subtopic"])
    .reset_index(drop=True)
)

print(f"Parent topics: {df[parent_topic_col].nunique()}")
print(f"Discovered topic paths: {df['topic_path'].nunique()}")
subtopic_summary_df

Parent topics: 5
Discovered topic paths: 20


,parent_topic,subtopic,topic_path,doc_count,avg_confidence,top_terms,split_applied
0,0,0,0.0,648,0.799467,"εαυτ, μεν, αλλα, αυτ, πασ, υμεισ, θεοσ, ποι, α...",True
1,0,1,0.1,247,0.800807,"παντ, εαυτ, λαλ, καθωσ, θεου, αυτ, ανθρωπ, δικ...",True
2,0,2,0.2,136,0.817519,"αυτ, ονοματι, εστιν, θεου, λεγ, δενδρ, καρπ, κ...",True
3,0,3,0.3,271,0.795842,"ειτε, αιων, θεω, αμην, παντ, δοξα, θεοσ, μαλλ,...",True
4,1,0,1.0,172,0.762997,"φαρισαι, αυτ, ταυτα, εγενετο, ειπαν, ποι, σαββ...",True
5,1,1,1.1,952,0.811713,"αυτ, εστιν, ιησ, ανθρωπ, κυρι, λεγ, παντ, ειπε...",True
6,1,2,1.2,165,0.753465,"νομ, εστιν, αμαρτι, ιουδαι, αβρααμ, αιματ, ιμα...",True
7,1,3,1.3,262,0.790177,"προφητ, αυτ, νομ, ειπεν, λεγ, κριν, παντ, κυρι...",True
8,2,0,2.0,181,0.760244,"εστιν, αληθει, θεου, θεοσ, αυτ, μεν, αλλ, κοσμ...",True
9,2,1,2.1,436,0.746932,"χριστ, θεου, ιησ, κυρι, εστιν, αλλ, ανθρωπ, χα...",True


In [22]:
topic_path_sizes = (
    df["topic_path"]
    .value_counts()
    .rename_axis("topic_path")
    .reset_index(name="doc_count")
)

top_per_subtopic = (
    df.sort_values("subtopic_confidence", ascending=False)
    .groupby("topic_path", as_index=False)
    .head(5)[["topic_path", "verse_ref", "text", "subtopic_confidence"]]
)

print("Largest subtopics:")
topic_path_sizes.head(15)

Largest subtopics:


,topic_path,doc_count
0,1.1,952
1,3.3,852
2,2.2,675
3,0.0,648
4,4.3,565
5,4.2,485
6,4.1,471
7,2.3,439
8,2.1,436
9,0.3,271


In [23]:
print("Representative verses by subtopic:")
top_per_subtopic.head(40)

Representative verses by subtopic:


,topic_path,verse_ref,text,subtopic_confidence
6932,3.3,mark 16:8,καὶ ἐξελθοῦσαι ταχὺ ἔφυγον ἀπὸ τοῦ μνημείου· ε...,0.967094
5233,3.3,matthew 2:13,Ἀναχωρησάντων δὲ αὐτῶν ἰδοὺ ἄγγελος κυρίου φαί...,0.966772
3491,4.3,john 8:14,Ἀπεκρίθη Ἰησοῦς καὶ εἶπεν αὐτοῖς· κἂν ἐγὼ μαρτ...,0.966560
7419,3.1,revelation 20:4,"Καὶ εἶδον θρόνους, καὶ ἐκάθισαν ἐπ᾽ αὐτούς, κα...",0.965823
7168,0.3,revelation 5:13,καὶ πᾶν κτίσμα ὃ ἐστιν ἐν τῷ οὐρανῷ καὶ ἐπὶ τῆ...,0.964856
7093,3.2,revelation 1:20,τὸ μυστήριον τῶν ἑπτὰ ἀστέρων οὓς εἶδες ἐπὶ τῆ...,0.964759
3982,4.2,john 20:19,Οὔσης οὖν ὀψίας τῇ ἡμέρᾳ ἐκείνῃ τῇ μιᾷ τῶν σ...,0.963145
3521,2.2,john 8:44,ὑμεῖς ἐκ τοῦ πατρὸς τοῦ διαβόλου ἐστὲ καὶ τὰς ...,0.963121
5585,0.0,matthew 12:45,τότε πορεύεται καὶ παραλαμβάνει μεθ᾽ ἑαυτοῦ ἑπ...,0.962934
4485,4.1,luke 9:33,Καὶ ἐγένετο ἐν τῷ διαχωρίζεσθαι αὐτοὺς ἀπ᾽ αὐτ...,0.962910
